# 安装必要库
llamafactory参考https://llamafactory.readthedocs.io/zh-cn/latest/getting_started/installation.html  
vllm安装参考https://vllm.hyper.ai/docs/getting-started/installation

## 模型选择与数据集
在实验时，我们经常使用1.5b小模型进行快速低成本实验，如果1.5b取得了好的效果，再复制到大模型上进行实验，这样做的好处就是可以快速验证想法，节约时间和 GPU资源成本。

本次实验我们使用的数据为**工具调用数据集**，该数据集包含约十万条由 Glaive AI(https://glaive.ai/)生成的关于工
具调用的对话样本，我们将数据集处理为多⻆色的多轮对话样本，包含用户（human）、模型（gpt）、工具调用 （function_call）和工具返回结果（observation）四种不同⻆色，同时还有一个工具列表（tools）字段，以 OpenAI 的格式(https://openai.com/index/function-calling-and-other-api-updates/)定义了可选工具。下面是数
据集中的一个样本示例



In [1]:
!export USE_MODELSCOPE_HUB=1

'export' �����ڲ����ⲿ���Ҳ���ǿ����еĳ���
���������ļ���


在Llama-Factory文件夹中，/examples下有各种配置文件示例，我们参考train_lora下的实例，编写我们模型的配置文件  qwen2_lora_sft.yaml ：

```yaml
### model
model_name_or_path: qwen/Qwen2-1.5B

### method
stage: sft
do_train: true
finetuning_type: lora
lora_target: all

### dataset
dataset: glaive_toolcall_en, glaive_toolcall_zh,alpaca_gpt4_en,alpaca_gpt4_zh
template: qwen
cutoff_len: 1024
max_samples: 50000
overwrite_cache: true
preprocessing_num_workers: 16

### output
output_dir: /root/autodl-tmp/checkpoints/agent
logging_steps: 100
save_steps: 1000
plot_loss: true
overwrite_output_dir: true

### train
per_device_train_batch_size: 1
gradient_accumulation_steps: 8
learning_rate: 1.0e-4
num_train_epochs: 3.0
lr_scheduler_type: cosine
warmup_ratio: 0.1
bf16: true
ddp_timeout: 180000000

### eval
val_size: 0.01
per_device_eval_batch_size: 1
eval_strategy: steps
eval_steps: 1000

```

In [2]:
!llamafactory-cli train ./code/qwen2_lora_sft.yaml

^C


[INFO|2026-01-08 23:06:14] llamafactory.launcher:143 >> Initializing 2 distributed tasks at: 127.0.0.1:57784


W0108 23:06:18.671000 22556 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.
W0108 23:06:18.842000 22556 site-packages\torch\distributed\run.py:803] 
W0108 23:06:18.842000 22556 site-packages\torch\distributed\run.py:803] *****************************************
W0108 23:06:18.842000 22556 site-packages\torch\distributed\run.py:803] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W0108 23:06:18.842000 22556 site-packages\torch\distributed\run.py:803] *****************************************
Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "C:\ProgramData\anaconda3\envs\vllm\Scripts\torchrun.exe\__main__.py", line 6, in <module>
  File "C:\Pr